In [2]:
import src.compute_payoff as compute_payoff
import numpy as np

In [ ]:
# update_rule[l,i,j,k] 
# l = 0 is the update to player's score when they are a donor  
# so update_rule[0,i,j,k] says "what is the update to the score of a player who has 
# reputation k when they do action j to a person with reputation i"
# l = 1 is the update to player's score when they are a receiver 
# so update_rule[1,i,j,k] says "what is the update to the score of a player who has 
# reputation i when they receive action j from a person with reputation k"

rng = np.random.default_rng(42)
strategies = rng.choice((0,1),size=(2,2,2))
strategy_x = strategies[0]
strategy_y = strategies[1]
update_rule = rng.choice((0,1,-1),size=(2,2,2,2)) 
update_rule[:,:,:,0] = np.where(update_rule[:,:,:,0] == -1, 0, update_rule[:,:,:,0])

print("big b update_rule")
print(update_rule)
print('strategy x')
print(strategy_x)
print('strategy y')
print(strategy_y)

big b update_rule
[[[[ 0  0]
   [ 1 -1]]

  [[ 0 -1]
   [ 0 -1]]]


 [[[ 1  0]
   [ 0  1]]

  [[ 1  1]
   [ 0 -1]]]]
strategy x
[[0 1]
 [1 0]]
strategy y
[[0 1]
 [0 1]]


In [ ]:
def realised_updates(focal_strategy, opponent_strategy, update_rule):
    """
    given focal player's strategy, other player's strategy, and an update rule, 
    return two matrices, donor_delta and receiver_delta.

    donor_delta tells us what will actually happens to the focal player's reputation score 
    when he is a donor. this depends on the focal player's strategy. more specifically,

    donor_delta[f, i] = score update to a reputation f focal player when he is a donor and meets a receiver with reputation i 

    receiver_delta tells us what will actually happent to the focal player's reputation score 
    when he is a receiver. this depends on the other player player's strategy. more specifcally,

    receiver_delta[f, k] = score update to a reputation f focal player when he is a receiver and meets a donor with reputation k

    
    """

    K = focal_strategy.shape[0] # number or repuation states 
    reps = np.arange(K)

    f_idx = reps[:, None]   # focal reputation, shape (K, 1)
    opp_idx = reps[None, :] # opponent reputation, shape (1, K)

    # donor case:
    # donor_delta[f, i] = update_rule[0, i, focal_strategy[i, f], f]
    donor_delta = update_rule[
        0,
        opp_idx,
        focal_strategy.T,
        f_idx,
    ]

    # receiver case:
    # receiver_delta[f, k] = update_rule[1, f, opponent_strategy[f, k], k]
    receiver_delta = update_rule[
        1,
        f_idx,
        opponent_strategy,
        opp_idx,
    ]

    return donor_delta, receiver_delta



In [100]:
reps = np.arange(2)

f_idx = reps[:, None]   # focal reputation, shape (K, 1)
opp_idx = reps[None, :] #

f_idx,opp_idx

(array([[0],
        [1]]),
 array([[0, 1]]))

In [92]:
def transition_vector_against(
    focal_strategy,
    opponent_strategy,
    update_rule,
    opponent_rep,
):
    """
    Return [alpha, beta, gamma] for the focal strategy
    against one opponent strategy type.

    alpha = P(0 -> 1 | focal currently has rep 0)
    beta  = P(1 -> 2 | focal currently has rep 1)
    gamma = P(1 -> 0 | focal currently has rep 1)

    This does NOT include:
    - rho_focal or rho_other
    - role factor 1/4
    - probability focal player currently has that reputation
    """

    donor_delta, receiver_delta = realised_updates(
        focal_strategy=focal_strategy,
        opponent_strategy=opponent_strategy,
        update_rule=update_rule,
    )

    # Which focal reputation state each transition is conditioned on
    focal_states = np.array([0, 1, 1])

    # Desired score change for alpha, beta, gamma
    changes = np.array([1, 1, -1])

    # Shape: (3, K)
    donor_match = donor_delta[focal_states, :] == changes[:, None]
    receiver_match = receiver_delta[focal_states, :] == changes[:, None]

    # Sum over opponent reputation only.
    # This is the "conditioned on Sx" part.
    trans_vector = (
        donor_match @ opponent_rep
        + receiver_match @ opponent_rep
    )

    return trans_vector

In [96]:
def transition_vector_for_type(
    focal_strategy,
    other_strategy,
    update_rule,
    focal_rep,
    other_rep,
    rho_focal,
    rho_other,
    role_factor=1 / 4,
    noise_val=0.0,
):
    """
    Return [alpha, beta, gamma] for one focal type
    in a two-strategy population.
    """

    trans_self = transition_vector_against(
        focal_strategy=focal_strategy,
        opponent_strategy=focal_strategy,
        update_rule=update_rule,
        opponent_rep=focal_rep,
    )

    trans_other = transition_vector_against(
        focal_strategy=focal_strategy,
        opponent_strategy=other_strategy,
        update_rule=update_rule,
        opponent_rep=other_rep,
    )

    trans = role_factor * (
        rho_focal * trans_self
        + rho_other * trans_other
    )

    trans = (1.0 - noise_val) * trans + noise_val / 2.0

    return trans

In [98]:
w_x = np.array([.1,.9])
w_y = np.array([.4,.6])

rho_x = .2
rho_y = .8

trans_x = transition_vector_for_type(
    focal_strategy=strategy_x,
    other_strategy=strategy_y,
    update_rule=update_rule,
    focal_rep=w_x,
    other_rep=w_y,
    rho_focal=rho_x,
    rho_other=rho_y,
)

alpha_x, beta_x, gamma_x = trans_x
alpha_x, beta_x, gamma_x

(np.float64(0.25),
 np.float64(0.12500000000000003),
 np.float64(0.37000000000000005))

In [99]:
trans_y = transition_vector_for_type(
    focal_strategy=strategy_y,
    other_strategy=strategy_x,
    update_rule=update_rule,
    focal_rep=w_y,
    other_rep=w_x,
    rho_focal=rho_y,
    rho_other=rho_x,
)

alpha_y, beta_y, gamma_y = trans_y
alpha_y, beta_y, gamma_y

(np.float64(0.25),
 np.float64(0.12500000000000003),
 np.float64(0.37000000000000005))

In [85]:
example = np.array(['a','b','c','d','e','f','g','h']).reshape((2,2,2))
example

array([[['a', 'b'],
        ['c', 'd']],

       [['e', 'f'],
        ['g', 'h']]], dtype='<U1')

In [87]:
reps = np.arange(2)
f_idx = reps[:, None]
strategy_x.T

array([[0, 1],
       [1, 0]])